## Hackathon: Get CO2 from MSE

### Setup Notes
I set up .venv in FUSIONAIHUB to work with all the packages above. You should be able to see it if you're in the FUSIONAIHUB directory and click on kernel.

Don't use the HDF5 file. Use the preprocessed data.

In [ ]:
import h5py
from joblib import dump, load
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn import preprocessing
import random

import matplotlib.pyplot as plt
import seaborn as sns

import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

In [ ]:
class FusionDataset(Dataset):
    
    def __init__(self, file_list, window_size=1):
        self.file_list = file_list
        self.window_size = window_size
        self.inputs = None
        self.targets = None
        self.setup()
        
    def setup(self):
        inputs = [None] * len(self.file_list)
        targets = [None] * len(self.file_list)
        missing_columns = self._identify_missing_columns()
        for k, f in enumerate(self.file_list):
            ece, mse = self._load_joblib_file(f, missing_columns)
            neighbors = NearestNeighbors(n_neighbors=1).fit(
                np.array(ece.index).reshape(-1, 1))
            distances, indices = neighbors.kneighbors(
                np.asarray(mse.index).reshape(-1, 1))
            indices = indices.flatten()
            ece = [ece.iloc[idx-self.window_size+1:idx+1] for idx in indices]
            inputs[k] = np.asarray(ece)
            targets[k] = np.asarray(mse)
        self.inputs = np.vstack(inputs)
        self.targets = np.vstack(targets)
        print(self.inputs.shape)
        print(self.targets.shape)
        
    def _identify_missing_columns(self):
        targets = []
        for file in self.file_list:
            targets.append(load(file)["mse"])
        df = pd.concat(targets)
        missing_columns = df.columns[df.isna().any()]
        return missing_columns
    
    def _load_joblib_file(self, file_name, missing_columns):
        print(file_name)
        data = load(file_name)
        ece_dataset = data["ece_cali"]
        mse_dataset = data["mse"].drop(missing_columns, axis=1, errors='ignore')
        return ece_dataset, mse_dataset
    
    def __getitem__(self, index):
        return (torch.tensor(self.inputs[index], dtype=torch.float32),
                torch.tensor(self.targets[index], dtype=torch.float32).unsqueeze(0))
        
    def __len__(self):
        return len(self.inputs)

In [ ]:
dir = Path("/scratch/gpfs/EKOLEMEN/hackathon/preprocessed/")
file_list = list(dir.glob("*.joblib"))

In [ ]:
dataset = FusionDataset(file_list, window_size=1)

In [ ]:
split_ratio = 0.8
train_dataset, test_dataset = random_split(dataset, [split_ratio, 1 - split_ratio])

In [ ]:
print('train_data length:', len(train_dataset))
print('test_data length:', len(test_dataset))

In [25]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

### Preprocessing

Do any preprocessing here if needed

data1 is smaller set input
data2 is bigger set output

### Dataset Module(s)

Goal: Trying to correlate 1 point for lower resolution with a window of data for higher resolution. So we want to be able to correlate the same time and find a fast consistent way to match the times

### Neural Network Modules

In [35]:
class Config:
    def __init__(self, input_dim, embed_dim, output_dim, hidden_dim, device = "cpu"):
        self.IN_DIM = input_dim
        self.EMBED_DIM = embed_dim
        self.OUT_DIM = output_dim
        self.HIDDEN = hidden_dim
        self.DEVICE = device

def build_model(in_dim, out_dim, layers, hidden, activation, normalize=lambda x: x):
    model = [normalize(nn.Linear(in_dim, hidden))]
    model += [activation()]
    for i in range(layers - 1):
        model += [normalize(nn.Linear(hidden, hidden))]
        model += [activation()]
    model += [normalize(nn.Linear(hidden, out_dim))]
    return nn.Sequential(*model)

class Decoder(nn.Module):

    def __init__(self, embed, hidden, out_dim, layers=2):
        super().__init__()
        self.fc1 = build_model(embed, hidden, layers, hidden, nn.ReLU)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, z):
        x = F.relu(self.fc1(z))
        return self.fc2(x), x

class Encoder(nn.Module):

    def __init__(self, in_dim, hidden, embed, layers=2):
        super().__init__()

        self.fc1 = nn.Linear(in_dim, hidden)
        self.encoder = build_model(hidden, embed, layers, hidden, nn.ReLU)

    def forward(self, x):
        embed = F.relu(self.fc1(x))
        return self.encoder(F.relu(embed))


In [36]:
class HackNet(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.encoder = Encoder(in_dim=config.IN_DIM, hidden=config.HIDDEN, embed=config.EMBED_DIM)
        self.decoder = Decoder(embed=config.EMBED_DIM, hidden=config.HIDDEN, out_dim=config.OUT_DIM)
        self._memory_unit = nn.GRU(config.EMBED_DIM, config.HIDDEN,
                                      num_layers= 2,
                                      batch_first=True)

        self.encoder = self.encoder.to(config.DEVICE)
        self.decoder = self.decoder.to(config.DEVICE)
        self._memory_unit = self._memory_unit.to(config.DEVICE)

    def forward(self, x): # B X T X IN_DIM

        embed = self.encoder(x) #B X T X EMBED_DIM
        mem_out = self._memory_unit(embed)[0]

        pred, _ = self.decoder(torch.cat([embed, mem_out]))  #B X T X OUT_DIM

        return pred

In [45]:
def train_epoch(network, optim, data_x, data_y):
    # data_x: (bs, window_size, x_dim); data_y: (bs, y_dim)

    dataset_size = data_x.shape[0]
    network.train()
    idxes = np.arange(0, dataset_size)
    random.shuffle(idxes)

    x = data_x[idxes]
    y = data_y[idxes]

    pred = network(x)
    #compute loss func
    loss = F.smooth_l1_loss(pred, y) #decide loss function, start with L1 distance
    #peform optimization
    optim.zero_grad()
    loss.backward()
    opt.step()

    return loss.item() #for logging


def val_epoch(network, optim, data_x, data_y):
    network.eval()
    dataset_size = data_x.shape[0]
    val_idxes = np.arange(0, dataset_size)
    random.shuffle(val_idxes)

    x = data_x[val_idxes]
    y = data_y[val_idxes]

    with torch.no_grad():

        #compute model output
        pred = network(x)
        #compute loss func
        loss = F.smooth_l1_loss(pred, y)

    return loss.item() #for logging


#training epoch code
config = Config(48, 64, 43, 64)
net = HackNet(config)
opt = torch.optim.Adam(net.parameters(), lr=3e-4)

num_epochs = 100

for epoch in range(num_epochs):

    for data_x, data_y in train_loader:
        train_loss = train_epoch(net, opt, data_x, data_y)
    
    for data_x, data_y in test_loader:
        val_loss = val_epoch(net, opt, data_x, data_y)
        break

/tmp/ipykernel_174944/2646173492.py:14: UserWarning: Using a target size (torch.Size([32, 1, 43])) that is different to the input size (torch.Size([64, 1, 43])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.smooth_l1_loss(pred, y) #decide loss function, start with L1 distance


RuntimeError: The size of tensor a (64) must match the size of tensor b (32) at non-singleton dimension 0